## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Success Critera Test

To deterime if the merge was successful without affecting the underlaying data integrity this test made and must pass before considering success.

Display a Dataframe with the following totals, filtered by Date + Reservations Status for the exported file.

---

## Filters
Filters
Date = `01-01-2026` - `01-31-2026`
Reservation Status = `CHECKEDOUT`

---

## Sum Values
Sum Col `Sold`
Sum Col `Room Revenue`

---

## Criteria
Total for `Sum` must equal **2235**
Total for `Room` Revenue must equal **175872**

# Connect

In [ ]:
# @title Connect to Google Drive {"vertical-output":true,"single-column":true,"display-mode":"code"}

from google.colab import drive
import os

def setup_environment(source_path, next_path):
    """
    Module: Setup Environment
    Mounts Google Drive, defines global directory variables, and ensures all
    required subfolders exist.
    """
    print("--- Initializing Environment ---")

    # 1. Mount Drive (with a safety catch)
    try:
        drive.mount('/content/drive', force_remount=True)
        print("v Drive mounted successfully.")
    except Exception as e:
        print(f"x Manual action required: Please click the Drive icon to mount. Error: {e}")

    # --- SETUP & AUTH ---
    # Use force_remount=True to attempt a fresh connection if it previously failed
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Drive mounted successfully.")
    except Exception as e:
        print(f"Manual action required: Please click the Drive icon in the left file pane to mount your drive. Error: {e}")

    # --- CONFIGURATION (Master Variables) ---
    global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

    SOURCE_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step01"}
    NEXT_DIR = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step02" # @param {"type":"string","placeholder":"/content/drive/Shareddrives/Client Hubs/Dovetail&Co/data_pipeline/process_step02"}

    NEW_DIR = os.path.join(SOURCE_DIR, "data_upload")
    PROCESSED_DIR = os.path.join(SOURCE_DIR, "data_processed")
    EXPORT_DIR = os.path.join(SOURCE_DIR, "data_export")
    FAILED_DIR = os.path.join(SOURCE_DIR, "data_failed")
    NEXT_DIR = os.path.join(NEXT_DIR, "data_upload")

    # Create the necessary folders if they don't exist
    if not os.path.exists(NEW_DIR):
        os.makedirs(NEW_DIR)
        print(f"Created directory: {NEW_DIR}")
    if not os.path.exists(PROCESSED_DIR):
        os.makedirs(PROCESSED_DIR)
        print(f"Created directory: {PROCESSED_DIR}")
    if not os.path.exists(EXPORT_DIR):
        os.makedirs(EXPORT_DIR)
        print(f"Created directory: {EXPORT_DIR}")
    if not os.path.exists(FAILED_DIR):
        os.makedirs(FAILED_DIR)
        print(f"Created directory: {FAILED_DIR}")

    # 4. Create directories dynamically if they don't exist
    directories = [NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR]

    for directory in directories:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"  -> Created directory: {directory}")

        print(f"v Setup complete. Checking for new files in: {NEW_DIR}\n")

# Process Data + Join Files

In [ ]:
import pandas as pd
import os
import glob
import re
from datetime import datetime

def validate_snap_date(date_str):
    """Validates if the extracted string is a viable YYYYMMDD date."""
    if not date_str or len(date_str) != 8 or not date_str.isdigit():
        return False

    year = int(date_str[:4])
    month = int(date_str[4:6])
    day = int(date_str[6:])

    if not (2000 <= year <= 2100):
        return False
    if not (1 <= month <= 12):
        return False
    if not (1 <= day <= 31):
        return False

    try:
        datetime.strptime(date_str, '%Y%m%d')
        return True
    except ValueError:
        return False

def extract_metadata(file_path):
    """Extracts PROPERTY_CODE and SNAP_DATE from the filename."""
    filename = os.path.basename(file_path)

    prop_code = "UNKNOWN"
    snap_date = None

    # Try to extract property code (e.g., NNNH) - looking for a 4-letter code
    prop_code_match = re.search(r'([A-Z]{4})', filename)
    if prop_code_match:
        prop_code = prop_code_match.group(1)

    # Try to extract date from the beginning (e.g., 20260723_NNNH...)
    date_start_match = re.match(r'^(\d{8})_', filename)
    if date_start_match:
        extracted_date = date_start_match.group(1)
        if validate_snap_date(extracted_date):
            snap_date = extracted_date

    # If not found at the beginning, try to extract date from the end (e.g., ..._20260716232505.csv)
    if snap_date is None:
        date_end_match = re.search(r'(\d{8})\d*\.csv$', filename)
        if date_end_match:
            extracted_date = date_end_match.group(1)
            if validate_snap_date(extracted_date):
                snap_date = extracted_date

    return prop_code, snap_date, validate_snap_date(snap_date)

def find_header_row(file_path, threshold=10):
    """Scans the start of a file to find the first row with significant columns."""
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if len(line.split(',')) > threshold:
                return i
    return 0

def load_latest_export(pattern_suffix, folder_path):
    """Finds and loads the latest file, dynamically locating the header row."""
    search_pattern = os.path.join(folder_path, f"*{pattern_suffix}*.csv")
    files = glob.glob(search_pattern)

    if not files:
        print(f"Warning: No files found for pattern {pattern_suffix} in {folder_path}")
        return None, None, None, None

    latest_file = max(files, key=os.path.getmtime)
    prop_code, snap_dt, valid_dt = extract_metadata(latest_file)

    # Dynamically find where the actual data starts to avoid ParserWarnings
    skip_count = find_header_row(latest_file)

    try:
        df = pd.read_csv(latest_file, skiprows=skip_count, sep=',', on_bad_lines='warn', engine='python')
        return df, prop_code, snap_dt, latest_file
    except Exception as e:
        print(f"Failed to parse {latest_file}: {e}")
        return None, prop_code, snap_dt, latest_file

# --- Execution Block ---
try:
    pms_df, pms_prop, pms_snap, resv_file = load_latest_export("_RESV_Export", NEW_DIR)
    stay_df, stay_prop, stay_snap, stay_file = load_latest_export("_STAY_Export", NEW_DIR)

    if pms_df is not None and stay_df is not None:
        PROPERTY_CODE = pms_prop
        SNAP_DATE = pms_snap

        print(f"Successfully processed files for Property: {PROPERTY_CODE}")
        print(f"Extracted Snap Date: {SNAP_DATE} (Valid: {validate_snap_date(SNAP_DATE)})")
        print(f"RESV File: {os.path.basename(resv_file)}")
        print(f"STAY File: {os.path.basename(stay_file)}")

        display(pms_df.head())
    else:
        print("Error: Loading failed. Check if files exist and have data rows.")

except Exception as e:
    print(f"An error occurred during processing: {e}")

Error: Loading failed. Check if files exist and have data rows.


In [ ]:
import pandas as pd
import io
import os
import glob
from datetime import datetime
from google.colab import drive
import pandas_gbq
import shutil

# --- EXECUTE SETUP ---
# setup_environment definition is located in the Preflight section
setup_environment(
    source_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01",
    next_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step02"
)

def load_latest_export(pattern_suffix, folder_path):
    """Finds and loads the latest file matching the suffix in the given folder."""
    search_pattern = os.path.join(folder_path, f"*{pattern_suffix}*.csv")
    files = glob.glob(search_pattern)
    if not files:
        print(f"⚠️ No files found for {pattern_suffix} in {folder_path}")
        return None, None, None, None

    latest_file = max(files, key=os.path.getmtime)
    prop_code, snap_dt, _ = extract_metadata(latest_file)
    print(f"\n--- Processing {os.path.basename(latest_file)} ---")

    try:
        with open(latest_file, "r", encoding="utf-8-sig", errors="replace", newline="") as file:
            lines = file.read().splitlines()

        header_index = -1
        anchors = ["Reservation Id", "Hotel Code", "Arrival Date", "Guest Id", "Lead Days", "Nights"]

        for index, line in enumerate(lines[:50]):
            matches = [anchor for anchor in anchors if anchor in line]
            if len(matches) >= 2:
                header_index = index
                break

        if header_index == -1:
            print(f"❌ Could not find CSV header in {os.path.basename(latest_file)}.")
            return None, None, None, None

        cleaned_csv_text = "\n".join(lines[header_index:])
        df = pd.read_csv(io.StringIO(cleaned_csv_text), skipinitialspace=True, on_bad_lines="warn", low_memory=False)

    except Exception as error:
        print(f"❌ Error parsing CSV: {error}")
        return None, None, None, None

    df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip().str.strip("'\"")
    object_columns = df.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        df[column] = df[column].astype("string").str.strip().str.strip("'\"")

    for col in ["Lead Days", "Nights"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")

    print(f"✅ Successfully loaded {len(df)} rows.")
    return df, prop_code, snap_dt, latest_file

# --- Execution & Export Block ---
if "NEW_DIR" in globals():
    pms_df, pms_prop, pms_snap, resv_file = load_latest_export("_RESV_Export", NEW_DIR)
    stay_df, stay_prop, stay_snap, stay_file = load_latest_export("_STAY_Export", NEW_DIR)

    if pms_df is not None and stay_df is not None:
        columns_to_extract = ['Reservation Id', 'Hotel Code', 'Arrival Date', 'Departure Date', 'Arrival Rate Code', 'Departure Room Type', 'Cancel Date', 'Created At Hz', 'First Night Price', 'Last Night Price', 'Lead Days', 'Nights', 'Origin Code', 'Postal Code', 'Purpose Of Stay', 'State']
        available_cols = [col for col in columns_to_extract if col in pms_df.columns]
        res_df_filtered = pms_df[available_cols]

        combined_df = pd.merge(stay_df, res_df_filtered, on=['Reservation Id', 'Hotel Code'], how='left', suffixes=('', '_res'))
        combined_df['PROPERTY_CODE'] = pms_prop
        combined_df['SNAP_DATE'] = pms_snap # This will be None if pms_snap was None

        output_filename = f"{pms_snap}_{pms_prop}_pms_data.csv"

        # Export to EXPORT_DIR and NEXT_DIR
        combined_df.to_csv(os.path.join(EXPORT_DIR, output_filename), index=False)
        combined_df.to_csv(os.path.join(NEXT_DIR, output_filename), index=False)
        print(f"\n✅ Merged data exported to: {output_filename}")

        # Move source files to PROCESSED_DIR
        for f_path in [resv_file, stay_file]:
            if os.path.exists(f_path):
                shutil.move(f_path, os.path.join(PROCESSED_DIR, os.path.basename(f_path)))
                print(f"✅ Source file moved to processed: {os.path.basename(f_path)}")
            else:
                print(f"⚠️ Warning: Source file not found, skipping move: {os.path.basename(f_path)}")

        display(combined_df.head())
    else:
        print("❌ Error: Loading failed. Check if files exist and have data rows.")
else:
    print("❌ Configuration Error: NEW_DIR not found.")

--- Initializing Environment ---
Mounted at /content/drive
v Drive mounted successfully.
Mounted at /content/drive
Drive mounted successfully.
v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload

v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload

v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload

v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload

v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload


--- Processing NNNH_JFKNOW_RESV_Export_Manual_20260723132117.csv ---
✅ Successfully loaded 41413 rows.

--- Processing NN

,Date,Confirmation Number,Rate Amount,Hotel Code,Original Rate Amount,Reservation Status,Room Type,Room No,Room Id,Block Count,...,First Night Price,Last Night Price,Lead Days,Nights,Origin Code,Postal Code,Purpose Of Stay,State,PROPERTY_CODE,SNAP_DATE
0,2024-12-09,100069.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,353.0,363397.0,NaN,...,152.15,152.15,280.0,4.0,<NA>,3385,NaN,<NA>,NNNH,20260723
1,2025-01-01,100000.0,100.0,NNNH,199.0,CANCELED,ADA,NaN,NaN,NaN,...,100.00,100.00,107.0,1.0,<NA>,<NA>,NaN,<NA>,NNNH,20260723
2,2025-01-02,100000.0,100.0,NNNH,139.0,CANCELED,ADA,NaN,NaN,NaN,...,100.00,100.00,107.0,1.0,<NA>,<NA>,NaN,<NA>,NNNH,20260723
3,2025-01-07,100197.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,348.0,363392.0,NaN,...,126.65,135.15,108.0,2.0,<NA>,68502,NaN,NE,NNNH,20260723
4,2025-01-08,100198.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,318.0,363363.0,NaN,...,116.00,135.15,94.0,3.0,<NA>,<NA>,NaN,<NA>,NNNH,20260723


In [ ]:
def run_master_pipeline():
    try:
        # Check if files were already processed in the current session by cell 1LOwz0Khke-X
        if 'pms_df' in globals() and 'stay_df' in globals() and pms_df is not None and stay_df is not None:
            print("✅ Data already loaded and merged in current session.")
            return True

        # Otherwise, try to load them manually
        RESV_PATTERN = 'NNNH_JFKNOW_RESV_Export'
        resv_df, _, _, _ = load_latest_export(RESV_PATTERN, NEW_DIR)

        STAY_PATTERN = 'NNNH_JFKNOW_STAY_Export'
        stay_df_check, _, _, _ = load_latest_export(STAY_PATTERN, NEW_DIR)

        if resv_df is not None and stay_df_check is not None:
            return True
        else:
            print("⚠️ No new files found in data_upload to process.")
            return False
    except Exception as e:
        print(f"Pipeline execution failed: {e}")
        return False

# Execute Pipeline and Capture Success State
pipeline_success = run_master_pipeline()

# Triggers the next notebook in the pipeline ONLY if the entire pipeline returned True
if pipeline_success:
    print("🚀 Success: Triggering Step 02...")
    try:
        get_ipython().run_line_magic('run', '"/content/drive/MyDrive/Colab Notebooks/StayInTouch/Step02_StayInTouch_AmemdPMSDataCRSData.ipynb"')
    except Exception as e:
        print(f"Could not trigger Step 02 notebook: {e}")
else:
    print("🛑 Failure/No Data: Downstream pipeline was NOT triggered.")

✅ Data already loaded and merged in current session.
🚀 Success: Triggering Step 02...
🔍 Searching for files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step01/data_upload
❌ Missing PMS file
❌ Missing CRS file

⚠️ Loading failed. Check your data_upload folder.
⚙️ Starting Enrichment Module...
❌ Enrichment Failed: pms_df or crs_df is missing. Please run the loading module successfully first.
📊 --- Merge Diagnostics ---
Total Rows: 177360
Successful CRS Matches: 98496 (55.53%)

Missing Data Heatmap (Top columns):
crs_channel       78864
crs_subsource    177360
crs_rate_type     78864
dtype: int64
--- Initializing Environment ---
Mounted at /content/drive
v Drive mounted successfully.
Mounted at /content/drive
Drive mounted successfully.
v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step02/data_upload

ၣ Starting Master Pipeline...
🔍 Searching for files in: /content/dr

,Date,Confirmation Number,Rate Amount,Hotel Code,Original Rate Amount,Reservation Status,Room Type,Room No,Room Id,Block Count,...,State,PROPERTY_CODE,SNAP_DATE,crs_confirm_no,channel_confirm_no,crs_channel_code,crs_channel,crs_subsource_code,crs_subsource,crs_rate_type
0,2024-12-09,100069.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,353.0,363397.0,NaN,...,NaN,NNNH,20260723,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown
1,2025-01-01,100000.0,100.0,NNNH,199.0,CANCELED,ADA,NaN,NaN,NaN,...,NaN,NNNH,20260723,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown
2,2025-01-02,100000.0,100.0,NNNH,139.0,CANCELED,ADA,NaN,NaN,NaN,...,NaN,NNNH,20260723,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown
3,2025-01-07,100197.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,348.0,363392.0,NaN,...,NE,NNNH,20260723,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown
4,2025-01-08,100198.0,0.0,NNNH,0.0,CHECKEDOUT,WSC,318.0,363363.0,NaN,...,NaN,NNNH,20260723,NaN,NaN,Unknown,Unknown,Unknown,Unknown,Unknown


၈ --- Post-Merge Column Verification ---


,Market Code,Source Code,Rate Amount,crs_channel
0,Discount,Direct,0.0,Unknown
1,Retail,Direct,100.0,Unknown
2,Retail,Direct,100.0,Unknown
3,Discount,Direct,0.0,Unknown
4,Discount,Direct,0.0,Unknown


- Sample values for Market Code: ['Discount' 'Retail' nan 'OTA' 'Complimentary']
- Sample values for Source Code: ['Direct' nan 'Expedia' 'Hopper' 'Trip']
- Sample values for Rate Amount: [  0.   100.   200.    92.88 105.  ]
- Sample values for crs_channel: ['Unknown' 'PMS' 'Expedia' 'Booking Engine' 'Booking.com']
၇ Exported merged file to: 20260723_NNNH_pms_data_merged.csv
ၦ Archived to Processed: 20260723_NNNH_pms_data.csv
ၦ Archived to Processed: 20260723_JFKNOW_crs_reservations.csv
🚀 Success: Triggering Step 03...
Successfully loaded revrebel_column_standardizer library.
--- Initializing Environment ---
Mounted at /content/drive
v Drive mounted successfully.
v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step03/data_upload

Found 1 files in /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step03/data_upload.

Processing: 20260723_NNNH_pms_data_merged.csv
  Applied renam

...

--- Running Success Criteria Test ---
❌ Error: Required column 'stay_date' not found in the dataset.
No files matching pattern found in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step05/data_export. Please run the master pipeline cell (zDkwBwlw-CyM) first.
Error: No processed file found to extract headers from. Please ensure the pipeline has run successfully.
Process complete. Run the verification cell above to load the file path.
--- Running Success Criteria Test ---
❌ Error: Required column 'stay_date' not found in the dataset.
📊 --- Post-Merge Column Verification ---
Detected verification columns: ['Market Code', 'Source Code', 'Rate Amount', 'crs_channel']


,Market Code,Source Code,Rate Amount,crs_channel
0,Discount,Direct,0.0,Unknown
1,Retail,Direct,100.0,Unknown
2,Retail,Direct,100.0,Unknown
3,Discount,Direct,0.0,Unknown
4,Discount,Direct,0.0,Unknown


- Sample values for Market Code: ['Discount' 'Retail' nan 'OTA' 'Complimentary']
- Sample values for Source Code: ['Direct' nan 'Expedia' 'Hopper' 'Trip']
- Sample values for Rate Amount: [  0.   100.   200.    92.88 105.  ]
- Sample values for crs_channel: ['Unknown' 'PMS' 'Expedia' 'Booking Engine' 'Booking.com']
--- Running Success Criteria Test ---
Filtering based on column: Date


,Metric,Actual,Target,Status
0,Sum Col 'Sold',2235.00,2235,✅ PASS
1,Sum Col 'Room Revenue',175872.66,175872,✅ PASS



✨ SUCCESS: All criteria met. Data integrity confirmed.


### Validation / Diagnostics
This section verifies the integrity of the merged data against the success criteria defined for January 2026.

In [ ]:
import pandas as pd
import os
import glob

# --- Validation Module ---
def run_success_criteria_test(df):
    print("--- Running Success Criteria Test ---")

    # Filtering strictly on the 'Date' column as requested
    date_col = 'Date'

    if date_col not in df.columns:
         print(f"❌ Error: Required column '{date_col}' not found in the dataset.")
         return

    print(f"Filtering based on column: {date_col}")
    df[date_col] = pd.to_datetime(df[date_col])

    # 1. Define Filters (Criteria uhfOwqak033W)
    start_date = '2026-01-01'
    end_date = '2026-01-31'
    status_filter = 'CHECKEDOUT'

    # 2. Apply Filters
    mask = (
        (df[date_col] >= start_date) &
        (df[date_col] <= end_date) &
        (df['Reservation Status'] == status_filter)
    )
    test_df = df.loc[mask].copy()

    # 3. Calculate Sums
    total_sold = pd.to_numeric(test_df['Sold'], errors='coerce').sum()
    total_revenue = pd.to_numeric(test_df['Room Revenue'], errors='coerce').sum()

    # 4. Success Criteria Targets
    target_sold = 2235
    target_revenue = 175872

    # 5. Display Results with Tolerance for Decimals
    sold_pass = int(round(total_sold)) == target_sold
    rev_pass = abs(total_revenue - target_revenue) < 1.0

    results_data = {
        "Metric": ["Sum Col 'Sold'", "Sum Col 'Room Revenue'"],
        "Actual": [round(total_sold, 2), round(total_revenue, 2)],
        "Target": [target_sold, target_revenue],
        "Status": [
            "✅ PASS" if sold_pass else "❌ FAIL",
            "✅ PASS" if rev_pass else "❌ FAIL"
        ]
    }

    results_df = pd.DataFrame(results_data)
    display(results_df)

    if sold_pass and rev_pass:
        print("\n✨ SUCCESS: All criteria met. Data integrity confirmed.")
    else:
        print("\n⚠️ WARNING: Criteria mismatch. Check if the dates or filters need adjustment.")

# Execution Logic
if 'combined_df' in locals():
    run_success_criteria_test(combined_df)
elif 'EXPORT_DIR' in globals():
    files = glob.glob(os.path.join(EXPORT_DIR, "*_pms_data.csv"))
    if files:
        latest_export = max(files, key=os.path.getmtime)
        print(f"Loading latest export for validation: {os.path.basename(latest_export)}")
        loaded_df = pd.read_csv(latest_export, low_memory=False)
        run_success_criteria_test(loaded_df)
    else:
        print("❌ Error: No exported files found in EXPORT_DIR.")
else:
    print("❌ Error: combined_df not found and EXPORT_DIR not defined.")

--- Running Success Criteria Test ---
Filtering based on column: Date


,Metric,Actual,Target,Status
0,Sum Col 'Sold',2235.00,2235,✅ PASS
1,Sum Col 'Room Revenue',175872.66,175872,✅ PASS



✨ SUCCESS: All criteria met. Data integrity confirmed.
